# Hawaii Beetles — Scale Bar Annotation Pipeline

This notebook downloads individual specimen images from the `imageomics/Hawaii-beetles` HuggingFace dataset and overlays a **1 mm scale bar** on each image, derived from the scale bar annotated in the corresponding group image.

**Requirements:**
- A HuggingFace account with access to `imageomics/Hawaii-beetles` (accept the dataset terms at https://huggingface.co/datasets/imageomics/Hawaii-beetles)
- A HuggingFace token (create one at https://huggingface.co/settings/tokens)

In [ ]:
# ── Cell 1 : Install dependencies ─────────────────────────────────────────────
%pip install huggingface_hub pandas opencv-python-headless Pillow requests -q

In [ ]:
# ── Cell 2 : Imports & HuggingFace login ──────────────────────────────────────
import ast
import io
import math
import os
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download, login, hf_hub_url
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings('ignore')

# ── Authenticate with HuggingFace ─────────────────────────────────────────────
# Option A – interactive prompt (recommended for first run)
login()   # will ask for your token; check add_to_git_credential=False if preferred

# Option B – paste your token directly (comment out login() above)

In [ ]:
# ── Cell 3 : Configuration ────────────────────────────────────────────────────
REPO_ID          = "imageomics/Hawaii-beetles"
REPO_TYPE        = "dataset"

# Paths inside the dataset repository
CSV_PATH         = "trait_annotations.csv"
GROUP_IMG_PREFIX = "group_images/"
INDIV_IMG_PREFIX = "individual_specimens/"

# Output directory for annotated images
OUTPUT_DIR       = Path("annotated_specimens")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Template matching confidence threshold
MATCH_THRESHOLD  = 0.6

In [ ]:
# ── Cell 4 : Load & inspect the annotations CSV ───────────────────────────────

csv_local = hf_hub_download(
    repo_id=REPO_ID,
    filename=CSV_PATH,
    repo_type=REPO_TYPE,
)

df = pd.read_csv(csv_local)

print(f"Dataset shape  : {df.shape}")
print(f"Columns        : {df.columns.tolist()}")
print(f"\ncm_scalebar unique values : {df['cm_scalebar'].unique()}")
print(f"Rows with cm=0 (to skip)  : {(df['cm_scalebar'] == 0.0).sum()}")
print(f"Rows with cm=1 (to process): {(df['cm_scalebar'] == 1.0).sum()}")
df.head(3)

In [ ]:
# ── Cell 5 : Helper functions ─────────────────────────────────────────────────

# ─── 5.1  Download an image from HF and return it as a NumPy BGR array ────────
def hf_load_image_bgr(repo_path: str) -> np.ndarray:
    """
    Stream a file from the HF dataset without saving it to disk.
    Returns a BGR NumPy array (OpenCV format).
    """
    api  = HfApi()
    url  = hf_hub_url(repo_id=REPO_ID, filename=repo_path, repo_type=REPO_TYPE)
    from huggingface_hub import get_token
    headers = {"Authorization": f"Bearer {get_token()}"}
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    arr  = np.frombuffer(resp.content, dtype=np.uint8)
    img  = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Could not decode image from {repo_path}")
    return img


# ─── 5.2  Parse coords_scalebar field ─────────────────────────────────────────
def parse_coords(raw) -> tuple[int, int, int, int]:
    """
    Accept any of these formats from the CSV:
      - "[x1, y1, x2, y2]"
      - "(x1, y1, x2, y2)"
      - "x1,y1,x2,y2"
    Returns (x1, y1, x2, y2) as ints.
    """
    intern = raw[2:-2]
    splited = intern.split(", ")
    x1, y1, x2, y2 = [int(float(v)) for v in splited]
    return x1, y1, x2, y2


# ─── 5.3  Scalebar pixel length in group image ────────────────────────────────
def scalebar_px_length(x1, y1, x2, y2) -> float:
    """Euclidean distance between the two scale-bar endpoints (pixels)."""
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# ─── 5.4  Template matching: locate individual image inside group image ────────
def find_individual_in_group(
    individual_bgr: np.ndarray,
    group_bgr: np.ndarray,
    threshold: float = MATCH_THRESHOLD,
) -> tuple[int, int] | None:
    """
    Uses cv2.matchTemplate (TM_CCOEFF_NORMED) to find the top-left corner
    of `individual_bgr` inside `group_bgr`.

    Returns (top_x, top_y) in group-image coordinates, or None if not found.
    """
    # Convert both images to grayscale for matching
    indiv_gray = cv2.cvtColor(individual_bgr, cv2.COLOR_BGR2GRAY)
    group_gray = cv2.cvtColor(group_bgr,      cv2.COLOR_BGR2GRAY)

    ih, iw = indiv_gray.shape
    gh, gw = group_gray.shape

    if ih > gh or iw > gw:
        raise ValueError(
            f"Individual image ({iw}×{ih}) is larger than group image ({gw}×{gh}). "
            "Cannot perform template matching."
        )

    result  = cv2.matchTemplate(group_gray, indiv_gray, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv2.minMaxLoc(result)

    if max_val < threshold:
        print(f"  ⚠  Template matching confidence {max_val:.3f} < {threshold}. "
              "Position may be unreliable.")

    return max_loc  # (top_x, top_y) in group image


# ─── 5.5  Draw the 1 mm scale bar on the individual image ─────────────────────
def draw_scale_bar(
    img_bgr: np.ndarray,
    px_per_mm: float,
    scale_factor : int = 2, # scale for the bar : base = 1 mm, now = scale_factor mm
    margin_frac: float = 0.04,   # margin from edges as fraction of image size
    bar_height_frac: float = 0.008, # bar thickness as fraction of image height
    font_scale_frac: float = 0.04,  # approx font height as fraction of image height
) -> np.ndarray:
    """
    Draws a horizontal 1 mm scale bar (white bar, black outline, white label)
    in the bottom-left corner of `img_bgr`.

    All sizes are derived from the image dimensions so the annotation looks
    consistent regardless of image resolution.

    Returns the annotated image (does NOT modify in-place).
    """
    out = img_bgr.copy()
    h, w = out.shape[:2]

    bar_length = int(px_per_mm * scale_factor)
    label = str(scale_factor) + " mm"

    bar_len_px  = int(round(bar_length))          # 1 mm in pixels
    bar_thick   = max(2, int(round(h * bar_height_frac)))
    margin      = max(8, int(round(min(h, w) * margin_frac)))

    # Font size: find the largest font scale that fits comfortably
    target_font_h = max(20, int(round(h * font_scale_frac)))
    font          = cv2.FONT_HERSHEY_SIMPLEX
    font_scale    = 0.1
    font_thick    = max(1, bar_thick // 2)
    
    for fs in [x * 0.1 for x in range(3, 50)]:
        (tw, th), _ = cv2.getTextSize(label, font, fs, font_thick)
        if th >= target_font_h:
            font_scale = fs
            break
    (text_w, text_h), baseline = cv2.getTextSize(label, font, font_scale, font_thick)

    # Position: bottom-left, above the margin
    bar_x1 = margin
    bar_x2 = bar_x1 + bar_len_px
    bar_y  = h - margin - text_h - baseline - bar_thick - margin // 2

    # Clamp to image bounds
    bar_x2 = min(bar_x2, w - margin)

    # Draw bar
    # cv2.rectangle(out, (bar_x1, bar_y - bar_thick), (bar_x1 + bar_thick, bar_y + bar_thick * 2), (255, 0, 0), -1)
    # cv2.rectangle(out, (bar_x2, bar_y - bar_thick), (bar_x2 - bar_thick, bar_y + bar_thick * 2), (255, 0, 0), -1)
    cv2.rectangle(out, (bar_x1, bar_y), (bar_x2, bar_y + bar_thick), (0, 0, 0), -1)

    # Draw label (black outline + white fill)
    text_x = bar_x1
    text_y = bar_y + bar_thick + baseline + text_h
    cv2.putText(out, label, (text_x, text_y), font, font_scale, (0, 0, 0),
                font_thick, cv2.LINE_AA)

    return out


print("✅ Helper functions defined.")

In [ ]:
# ── Cell 6 : Core pipeline function ──────────────────────────────────────────

def process_individual_specimen(row: pd.Series, output_dir: Path = OUTPUT_DIR) -> bool:
    """
    Full pipeline for one specimen row from trait_annotations.csv.

    Steps:
      1. Skip if cm_scalebar == 0.0
      2. Download individual image from HF
      3. Fetch group image from HF (not saved to disk)
      4. Parse scale-bar coordinates in the group image
      5. Compute pixels-per-mm from the group image scale bar
      6. Locate the individual crop inside the group image (template matching)
      7. Verify scale bar falls outside (or at edge of) the individual crop
      8. Draw 1 mm scale bar on the individual image
      9. Save annotated image

    Returns True on success, False if skipped / failed.
    """
    individual_id = row["individualID"]
    individual_pos = row["BeetlePosition"]
    cm_scalebar   = float(row["cm_scalebar"])

    # ── Step 1 : skip specimens with no valid scale bar ────────────────────────
    if cm_scalebar == 0.0:
        print(f"  ⏭  Skipping {individual_id}  (cm_scalebar = 0.0)")
        return False

    # Resolve HF repository paths
    # individual image filename follows the pattern: IMG_..._<individualID>.png
    indiv_filename   = _resolve_individual_filename(individual_id, individual_pos)
    indiv_repo_path  = INDIV_IMG_PREFIX + indiv_filename

    group_filename   = str(row["groupImageFilePath"]).strip()
    # groupImageFilePath may already include the subfolder; normalise it
    if not group_filename.startswith("HawaiiBeetles"):
        group_repo_path = GROUP_IMG_PREFIX + Path(group_filename).name
    else:
        group_repo_path = group_filename

    print(f"  Processing : {individual_id}")

    # ── Step 2 : Download individual image ────────────────────────────────────
    try:
        indiv_bgr = hf_load_image_bgr(indiv_repo_path)
    except Exception as e:
        print(f"  ❌ Could not download individual image: {e}")
        return False
    ih, iw = indiv_bgr.shape[:2]
    # print(f"  Indiv size : {iw} × {ih} px")

    # ── Step 3 : Fetch group image (not saved) ────────────────────────────────
    try:
        group_bgr = hf_load_image_bgr(group_repo_path)
    except Exception as e:
        print(f"  ❌ Could not fetch group image: {e}")
        return False
    gh, gw = group_bgr.shape[:2]
    # print(f"  Group size : {gw} × {gh} px")

    # ── Step 4 : Parse scale bar coords ──────────────────────────────────────
    try:
        x1, y1, x2, y2 = parse_coords(row["coords_scalebar"])
    except Exception as e:
        print(f"  ❌ Could not parse coords_scalebar: {e}")
        return False
    # print(f"  Scale bar coords (group): ({x1},{y1}) → ({x2},{y2})")

    # ── Step 5 : Compute pixels per mm ───────────────────────────────────────
    sb_px        = scalebar_px_length(x1, y1, x2, y2)   # pixels for cm_scalebar cm
    px_per_cm    = sb_px / cm_scalebar
    px_per_mm    = px_per_cm / 10.0
    # print(f"  Scale bar  : {sb_px:.1f} px = {cm_scalebar} cm  →  {px_per_mm:.2f} px/mm")

    if px_per_mm < 1:
        print("  ❌ Computed px/mm < 1; scale bar coordinates seem wrong. Skipping.")
        return False

    # ── Step 6 : Locate individual crop in group image ────────────────────────
    try:
        top_x, top_y = find_individual_in_group(indiv_bgr, group_bgr)
    except Exception as e:
        print(f"  ❌ Template matching failed: {e}")
        return False
    # print(f"  Individual crop position in group: top-left = ({top_x}, {top_y})")

    # ── Step 7 : Draw 1 mm scale bar on the individual image ──────────────────
    annotated = draw_scale_bar(indiv_bgr, px_per_mm)

    # ── Step 8 : Save ─────────────────────────────────────────────────────────
    out_path = output_dir / indiv_filename
    cv2.imwrite(str(out_path), annotated)
    # print(f"  ✅ Saved → {out_path}")
    return True


# ─── Utility: resolve the filename of an individual specimen image ─────────────
def _resolve_individual_filename(individual_id: str, beetle_position:int) -> str:
    """
    The individual image filename has the pattern IMG_0535_specimen_4_TREOBT_NEON.BET.D20.002341.png
    where individual_id looks like NEON.BET.D20.XXXXXX.
    We find the matching filename by listing the HF repository tree.
    Results are cached in a module-level dict to avoid repeated API calls.
    """
    if not hasattr(_resolve_individual_filename, "_cache"):
        _resolve_individual_filename._cache = {}
        api   = HfApi()
        files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)
        for f in files:
            if f.startswith(INDIV_IMG_PREFIX):
                fname = Path(f).name          # e.g. IMG_0001_NEON.BET.D20.123456.png
                # extract the individualID part (everything after last underscore group)
                # Pattern: IMG_<digits>_<individualID>.png
                stem = Path(fname).stem       # IMG_0001_NEON.BET.D20.123456
                parts = stem.split("_")    # ['IMG', '0001', 'NEON.BET.D20.123456']
                if len(parts) == 6:
                    _resolve_individual_filename._cache[parts[-1]] = fname

    cache = _resolve_individual_filename._cache
    if individual_id not in cache:
        raise FileNotFoundError(
            f"No individual image file found for individualID='{individual_id}' "
            f"in {INDIV_IMG_PREFIX}. Available IDs (first 5): "
            f"{list(cache.keys())[:5]}"
        )
    return cache[individual_id]


print("✅ Pipeline function defined.")

In [ ]:
# ── Cell 7 : TEST — process the FIRST valid individual image only ─────────────
#
# This cell runs the full pipeline on a single specimen so you can inspect the
# result before running the full dataset.
#
from IPython.display import display
import matplotlib.pyplot as plt

# Find the first row with a valid scale bar
test_row = df[df["cm_scalebar"] == 1.0].iloc[0]
print("Test specimen:")
print(test_row.to_dict())
print()

success = process_individual_specimen(test_row)

if success:
    # ── Display the annotated image inline ──────────────────────────────────
    indiv_filename = _resolve_individual_filename(test_row["individualID"], test_row["BeetlePosition"])
    out_path       = OUTPUT_DIR / indiv_filename
    annotated_bgr  = cv2.imread(str(out_path))
    annotated_rgb  = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(annotated_rgb)
    ax.set_title(f"Annotated: {test_row['individualID']}\n(scale bar = 1 mm)", fontsize=11)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"\n✅ Test passed. Annotated image saved to: {out_path}")
else:
    print("❌ Test failed — check the messages above.")

In [ ]:
# ── Cell 8 : Full pipeline — process all individual specimens ─────────────────
#
# ⚠ This will download and process ALL specimens. It may take a long time
#   depending on the number of images and your connection speed.
#
# Run Cell 7 first to verify the pipeline works correctly.
import time 

results = {"success": 0, "skipped": 0, "failed": 0}

T_start = time.time()
count = 0
for idx, row in df.iterrows():
    count += 1
    try:
        ok = process_individual_specimen(row)
        if ok:
            results["success"] += 1
        else:
            results["skipped"] += 1
    except Exception as exc:
        print(f"  ❌ Unhandled error for row {idx} ({row.get('individualID', '?')}): {exc}")
        results["failed"] += 1
    elapsed = round(time.time() - T_start, 1)
    speed = round(elapsed/count, 2)
    print("img done : ", count, "/ time elapsed : ", elapsed, "/ second per image : ", speed, "/ estimated time left (min): ", speed * (1600 - count) // 60)

print("\n" + "═" * 60)
print("Pipeline complete")
print(f"  ✅ Processed : {results['success']}")
print(f"  ⏭  Skipped   : {results['skipped']}  (cm_scalebar = 0.0)")
print(f"  ❌ Failed    : {results['failed']}")
print(f"\nAnnotated images saved in: {OUTPUT_DIR.resolve()}")